# AdCanvas 서브 노트북 (평가용)

메인 노트북과 같은 파이프라인으로, **같은 사진·같은 생성 설정에서 단계를 하나씩 바꿔** 비교한다.

| 조건 | 분할 | 프롬프트 | 보는 것 | 담당 |
|---|---|---|---|---|
| **A. 분할만 변경** | RMBG-2.0 / DINO+SAM / DINO+SAM 파인튜닝 | 고정 | 누끼가 결과를 얼마나 바꾸나 | 누끼 → 생성 |
| **B. 프롬프트만 변경** | 파인튜닝 고정 | 요청 그대로 / VLM 해석 | VLM이 요청 반영을 얼마나 높이나 | VLM → 생성 |
| **C. 전체 파이프라인** | 파인튜닝 | VLM | 최종 품질 (원본 음식 재합성 켬/끔) | 생성 |

| 평가 항목 | 담당 섹션 |
|---|---|
| ① 음식 마스크 품질 (IoU, 경계 BF, MAE) | 1. 누끼 |
| ② 픽셀 보존 | 3. 배경 생성 |
| ③ 요청 반영 | 2. VLM |
| ④ 시각적 품질 (사람 평가) | 4. 종합 |

**구성**: 0~2 공통(설치, 설정, 평가 세트) → 1. 누끼 → 2. VLM → 3. 배경 생성 → 4. 종합
* 각 팀은 자기 섹션만 채움. 결과는 드라이브 `eval/` 아래 정해진 폴더에 저장해서 다음 팀이 읽음

# 0. 설치 (공통)

In [ ]:
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q pillow-heif timm kornia einops

# 1. 설정 (공통)
* `EVAL_GEN`: **모든 조건이 같이 쓰는 생성 설정** (생성팀이 확정). 조건끼리 비교할 때 바꾸지 않음
* `USE_RMBG`: RMBG-2.0 비교를 할지 (허깅페이스 토큰이 코랩 보안 비밀 `HF_TOKEN`에 있어야 함)

In [ ]:
import os, glob, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy import ndimage

from google.colab import drive
drive.mount('/content/drive')

NUKKI_BASE = '/content/drive/MyDrive/AdCanvas_nukki'
NUKKI_WEIGHTS = f'{NUKKI_BASE}/weights/sam2_ft.pt'
IMG_DIR = f'{NUKKI_BASE}/dataset/images'
GT_DIR = f'{NUKKI_BASE}/dataset/gt'            # 흑백 정답 (gt/test: 평가용 20장)
META = f'{NUKKI_BASE}/dataset/meta.csv'        # 사진 분류 (multi, white, hand_tray, normal)
EVAL_DIR = f'{NUKKI_BASE}/eval'                # 평가 결과 (조건별 마스크, 점수, 그림)
os.makedirs(EVAL_DIR, exist_ok=True)

EVAL = dict(split='test', eval_side=1024, bf_tol=0.0075)
EVAL_GEN = dict(seed=42, ratio='1:1', long_side=1024, steps=24)   # 생성팀이 확정
USE_RMBG = True

NUKKI_CONDS = ['rmbg', 'dino_sam', 'dino_sam_ft']     # A조건: 분할만 변경

# 2. 평가 세트 (공통)
* 정답이 있는 test 사진 + 분류
* `cases.csv`: 사진마다 **기본 요청 1개 + 상충 요청 1개** (처음 실행할 때 틀을 만들어 줌 → 드라이브에서 요청 문장을 고쳐서 씀)
  * 상충 예: 흰 음식 + 하얀 배경 / 손에 든 음식 + 테이블 위 / 여러 그릇 + 한 음식만 강조

In [ ]:
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')
IMAGES = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{IMG_DIR}/*/*') if p.lower().endswith(EXTS)}
NAMES = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{GT_DIR}/{EVAL['split']}/*.png"))
NAMES = [n for n in NAMES if n in IMAGES]
CAT = dict(zip(*[pd.read_csv(META)[c] for c in ('image', 'category')])) if os.path.exists(META) else {}

CONFLICT = {   # 분류별 상충 요청 예시 (자유롭게 고치기)
    'white': ('white_on_white', 'clean white marble table, bright white background'),
    'hand_tray': ('hand_vs_table', 'placed on a rustic wooden dining table, no hands'),
    'multi': ('single_focus', 'highlight only the main dish, minimal background'),
    'normal': ('style_clash', 'neon cyberpunk street at night'),
}
CASES = f'{EVAL_DIR}/cases.csv'
if not os.path.exists(CASES):
    rows = []
    for n in NAMES:
        c = CAT.get(n, 'normal')
        rows.append({'case': f'{n}_base', 'image': n, 'category': c, 'conflict': 'none',
                     'request': 'warm cozy restaurant table, soft natural light, food photography'})
        kind, req = CONFLICT.get(c, CONFLICT['normal'])
        rows.append({'case': f'{n}_{kind}', 'image': n, 'category': c, 'conflict': kind, 'request': req})
    pd.DataFrame(rows).to_csv(CASES, index=False, encoding='utf-8-sig')
cases = pd.read_csv(CASES)
print(f"평가 사진 {len(NAMES)}장 | 분류 {pd.Series([CAT.get(n, '?') for n in NAMES]).value_counts().to_dict()}")
print(f'평가 케이스 {len(cases)}개 → {CASES}')

# 1. 누끼 평가 — 누끼팀

## 1-1. 누끼 모델 (메인 노트북과 같은 코드)
* 원래 SAM(`dino_sam`)과 파인튜닝 SAM(`dino_sam_ft`)을 둘 다 불러서 바꿔 가며 씀

In [ ]:
import os

import cv2
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy import ndimage

SAM_MODEL = 'facebook/sam2.1-hiera-small'
GD_ID = 'IDEA-Research/grounding-dino-tiny'
MAX_SIDE = 2048            # 이보다 큰 사진은 줄여서 처리하고, 마스크는 원래 크기로 되돌림

# 1_pipeline.ipynb와 같은 값 (11/16 test IoU 0.906, BF 0.827)
DINO_CFG = dict(
    dish_words=['plate', 'bowl', 'dish', 'pot'],            # 그릇 (기본 선택)
    other_words=['cup', 'glass', 'spoon', 'chopsticks'],    # 후보로만 (기본 선택 해제)
    food_words=['food'],                                    # 그릇 없이 놓인 음식
    box_th=0.3, nms_iou=0.6, max_area=0.9, food_max_area=0.5,
)
CAND = dict(
    utensil_words=['spoon', 'chopsticks'],
    utensil_overlap=0.2,   # 수저의 20% 이상이 그릇 외곽 안에 있으면 걸친 것 (원래 SAM일 때만 합침)
    part_min=0.1,          # 마스크 조각 중 가장 큰 조각의 10% 이상만 남김
    solidity=0.9,          # 원래 SAM일 때: 그릇이 볼록 외곽의 90% 미만이면 외곽으로 채움
    inside_drop=0.9,       # 다른 그릇 안에 90% 이상 들어간 후보는 중복으로 제거
    sam_chunk=4,           # SAM을 박스 몇 개씩 돌릴지 (메모리 절약)
)

_m = {}   # 불러온 모델: device, gd_proc, gd, sam, finetuned


# ---------------- 불러오기 ----------------
def _device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def load(weights=NUKKI_WEIGHTS):
    """모델 불러오기 (서버 켤 때 한 번). 반환: 파인튜닝 가중치를 썼는지"""
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    try:
        from pillow_heif import register_heif_opener
        register_heif_opener()   # 아이폰 HEIC
    except ImportError:
        pass
    dev = _device()
    _m['device'] = dev
    _m['gd_proc'] = AutoProcessor.from_pretrained(GD_ID)
    _m['gd'] = AutoModelForZeroShotObjectDetection.from_pretrained(GD_ID).to(dev).eval()
    sam = SAM2ImagePredictor.from_pretrained(SAM_MODEL, device=dev)
    _m['finetuned'] = bool(weights) and os.path.exists(weights)
    if _m['finetuned']:
        res = sam.model.load_state_dict(torch.load(weights, map_location=dev), strict=False)
        assert not res.missing_keys and not res.unexpected_keys, 'sam2_ft.pt와 SAM_MODEL이 맞지 않음'
    _m['sam'] = sam
    _m['sam_img'] = None   # set_image 한 사진 (클릭 보정 때 다시 계산하지 않으려고)
    return _m['finetuned']


def _ready():
    if 'sam' not in _m:
        load()


def _prep(img):
    """PIL 사진 → (처리용 RGB 사진, 원래 크기). EXIF 회전 반영, 긴 변 MAX_SIDE로"""
    img = ImageOps.exif_transpose(img).convert('RGB')
    size = img.size
    if max(size) > MAX_SIDE:
        img = img.copy()
        img.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
    return img, size


def _to_size(m, size):
    """bool 마스크 → 원래 크기의 흑백 PIL (L, 0/255)"""
    out = Image.fromarray((m * 255).astype(np.uint8))
    return out if out.size == size else out.resize(size, Image.NEAREST)


def _set_image(img_np):
    key = (img_np.shape, int(img_np[::97, ::89].sum()))
    if _m['sam_img'] != key:
        _m['sam'].set_image(img_np)
        _m['sam_img'] = key


# ---------------- DINO ----------------
def _box_area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


@torch.no_grad()
def dino_boxes(img):
    """사진 → [{'box', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택)}]"""
    from torchvision.ops import nms
    C, proc, gd = DINO_CFG, _m['gd_proc'], _m['gd']
    words = C['dish_words'] + C['other_words'] + C['food_words']
    kinds = ['dish'] * len(C['dish_words']) + ['other'] * len(C['other_words']) + ['food'] * len(C['food_words'])
    inputs = proc(images=img, text='. '.join(words) + '.', return_tensors='pt').to(_m['device'])
    out = gd(**inputs)
    probs = out.logits[0].sigmoid().float().cpu()
    ids = inputs['input_ids'][0].tolist()
    cols = []
    for w in words:
        sub = proc.tokenizer(w, add_special_tokens=False)['input_ids']
        pos = [i + j for i in range(len(ids) - len(sub) + 1) if ids[i:i + len(sub)] == sub for j in range(len(sub))]
        cols.append(probs[:, pos].max(1).values if pos else torch.zeros(len(probs)))
    score, best = torch.stack(cols, 1).max(1)
    W, H = img.size
    cx, cy, bw, bh = out.pred_boxes[0].float().cpu().unbind(-1)
    boxes = torch.stack([(cx - bw / 2) * W, (cy - bh / 2) * H, (cx + bw / 2) * W, (cy + bh / 2) * H], -1)
    boxes[:, 0::2] = boxes[:, 0::2].clamp(0, W)
    boxes[:, 1::2] = boxes[:, 1::2].clamp(0, H)
    keep = [i for i in range(len(boxes)) if score[i] > C['box_th'] and _box_area(boxes[i].tolist()) <
            (C['food_max_area'] if kinds[int(best[i])] == 'food' else C['max_area']) * W * H]
    if not keep:
        return []
    keep = torch.tensor(keep)
    k = keep[nms(boxes[keep], score[keep], C['nms_iou'])].tolist()
    return [{'box': boxes[i].tolist(), 'label': words[int(best[i])], 'score': float(score[i]),
             'kind': kinds[int(best[i])], 'dish': kinds[int(best[i])] != 'other'} for i in k]


# ---------------- 마스크 도우미 ----------------
def _box_mask(shape, box, expand=0.1):
    x0, y0, x1, y1 = box
    ex, ey = expand * (x1 - x0), expand * (y1 - y0)
    out = np.zeros(shape, bool)
    out[max(int(y0 - ey), 0):max(int(np.ceil(y1 + ey)), 0), max(int(x0 - ex), 0):max(int(np.ceil(x1 + ex)), 0)] = True
    return out


def _local(fn, m, pad=0):
    out = np.zeros_like(m)
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return out
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1])
    return out


def _hull(m):
    def f(c):
        out = np.zeros(c.shape, np.uint8)
        cv2.fillConvexPoly(out, cv2.convexHull(cv2.findNonZero(c.astype(np.uint8))), 1)
        return out.astype(bool)
    return _local(f, m)


def _close_fill(m):
    return _local(lambda c: ndimage.binary_fill_holes(ndimage.binary_closing(c, iterations=3)), m, pad=4)


def _keep_parts(c, box):
    lab, n = ndimage.label(c)
    if n <= 1:
        return c
    idx = np.arange(1, n + 1)
    sizes = ndimage.sum(c, lab, idx)
    inside = ndimage.sum(_box_mask(c.shape, box), lab, idx)
    keep = idx[(sizes >= CAND['part_min'] * sizes.max()) & (inside >= 0.5 * sizes)]
    return np.isin(lab, keep if len(keep) else idx[[sizes.argmax()]])


def _crop(m, box, fn, pad=4):
    ys, xs = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
    if not len(ys):
        return m
    y0, y1 = max(ys[0] - pad, 0), min(ys[-1] + 1 + pad, m.shape[0])
    x0, x1 = max(xs[0] - pad, 0), min(xs[-1] + 1 + pad, m.shape[1])
    out = np.zeros_like(m)
    out[y0:y1, x0:x1] = fn(m[y0:y1, x0:x1], [box[0] - x0, box[1] - y0, box[2] - x0, box[3] - y0])
    return out


def _pick_whole(ms, ss, box, mm_score=0.6, mm_inside=0.9):
    ok = [j for j in range(len(ms)) if ms[j].any() and ss[j] >= mm_score * ss.max()]
    bm = _box_mask(ms[0].shape, box)
    ok = [j for j in ok if (ms[j] & bm).sum() >= mm_inside * ms[j].sum()]
    return ms[max(ok, key=lambda j: ms[j].sum())] if ok else ms[int(ss.argmax())]


def _clean_dish(m, box):
    out = _crop(m, box, lambda c, b: ndimage.binary_fill_holes(ndimage.binary_closing(_keep_parts(c, b), iterations=3)))
    if not out.any():
        return out
    h = _hull(out) & (_box_mask(m.shape, box, 0.05) | out)
    return h if out.sum() < CAND['solidity'] * h.sum() else out


# ---------------- SAM + 후처리 ----------------
@torch.no_grad()
def _sam_masks(img_np, boxes, kinds):
    """파인튜닝 SAM은 마스크 1개, 원래 SAM은 3개 중 전체 물체를 고름"""
    _set_image(img_np)
    H, W = img_np.shape[:2]
    multi = not _m['finetuned']
    out = []
    for s0 in range(0, len(boxes), CAND['sam_chunk']):
        b = np.array(boxes[s0:s0 + CAND['sam_chunk']], np.float32)
        m, sc, _ = _m['sam'].predict(box=b, multimask_output=multi)
        m = np.asarray(m).reshape(len(b), -1, H, W) > 0.5
        if not multi:
            out += list(m[:, 0])
            continue
        sc = np.asarray(sc).reshape(len(b), -1)
        for k in range(len(b)):
            out.append(m[k, int(sc[k].argmax())] if kinds[s0 + k] == 'other' else _pick_whole(m[k], sc[k], boxes[s0 + k]))
    return out


def _tidy(cands, light):
    for c in cands:
        if c['dish'] and c['kind'] != 'food':
            c['mask'] = _close_fill(_crop(c['mask'], c['box'], _keep_parts, pad=0)) if light else _clean_dish(c['mask'], c['box'])
    dishes = sorted([c for c in cands if c['dish'] and c['kind'] != 'food'], key=lambda c: -c['mask'].sum())
    for i, small in enumerate(dishes):
        for big in dishes[:i]:
            if 'dup_of' not in big and (small['mask'] & big['mask']).sum() >= CAND['inside_drop'] * max(small['mask'].sum(), 1):
                small['dish'], small['dup_of'] = False, big['id']
                break
    kept = [d for d in dishes if 'dup_of' not in d]
    union = np.zeros(cands[0]['mask'].shape, bool)
    for d in kept:
        union |= d['mask']
    for f in [c for c in cands if c['kind'] == 'food']:
        f['mask'] = _close_fill(_crop(f['mask'], f['box'], _keep_parts, pad=0))
        a = f['mask'].sum()
        swallows = any((f['mask'] & d['mask']).sum() > 0.3 * d['mask'].sum() for d in kept)
        if a == 0 or (f['mask'] & union).sum() >= 0.5 * a or swallows:
            f['dish'], f['dup_of'] = False, -1
    utensils = [u for u in cands if u['label'] in CAND['utensil_words'] and u['mask'].any()]
    if utensils and not light:
        hulls = {c['id']: _hull(c['mask']) for c in cands if c['dish'] and 'dup_of' not in c and c['mask'].any()}
        for u in utensils:
            if not hulls:
                break
            best = max(hulls, key=lambda i: (u['mask'] & hulls[i]).sum())
            if (u['mask'] & hulls[best]).sum() > CAND['utensil_overlap'] * u['mask'].sum():
                d = next(c for c in cands if c['id'] == best)
                d['mask'] = _close_fill(d['mask'] | u['mask'])
                u['merged_into'] = best
    return [c for c in cands if 'dup_of' not in c]


# ---------------- 바깥에서 쓰는 함수 ----------------
def candidates(img):
    """사진 → 그릇 후보 목록. 박스는 원래 사진 좌표, mask는 원래 크기의 흑백 PIL
    [{'id', 'label', 'score', 'kind'(dish/other/food), 'dish'(기본 선택), 'box'[x0,y0,x1,y1], 'mask'}]
    그릇을 하나도 못 찾으면 [] (서비스에서는 사용자가 직접 박스를 그리게 → refine)"""
    _ready()
    work, size = _prep(img)
    found = dino_boxes(work)
    if not any(f['dish'] for f in found):
        return []
    masks = _sam_masks(np.array(work), [f['box'] for f in found], [f['kind'] for f in found])
    for i, (f, m) in enumerate(zip(found, masks)):
        f.update(id=i, mask=m)
    cands = _tidy(found, light=_m['finetuned'])
    s = size[0] / work.size[0]
    return [{'id': c['id'], 'label': c['label'], 'score': round(c['score'], 3), 'kind': c['kind'], 'dish': c['dish'],
             'box': [round(v * s, 1) for v in c['box']], 'mask': _to_size(c['mask'], size)} for c in cands]


def merge(img, cands, ids=None):
    """후보 중 고른 것만 합친 마스크 (ids=None이면 기본 선택된 그릇 전부)"""
    W, H = ImageOps.exif_transpose(img).size
    out = np.zeros((H, W), bool)
    for c in cands:
        if (c['id'] in ids) if ids is not None else c['dish']:
            out |= np.asarray(c['mask']) > 127
    return _to_size(out, (W, H))


def segment(img):
    """팀 공통 함수 모양 (KAN-5): RGB 사진 → 흑백 마스크 (L, 사진과 같은 크기, 255 = 음식+그릇)"""
    return merge(img, candidates(img))


@torch.no_grad()
def refine(img, box, points=None, labels=None):
    """클릭 보정: 박스(원래 사진 좌표) + 클릭 점(1 포함, 0 제외) → 그 그릇 하나의 마스크 (L, 원래 크기)"""
    _ready()
    work, size = _prep(img)
    s = work.size[0] / size[0]
    _set_image(np.array(work))
    kw = dict(box=np.array([v * s for v in box], np.float32), multimask_output=False)
    if points:
        kw.update(point_coords=np.array(points, np.float32) * s, point_labels=np.array(labels, np.int32))
    m, _, _ = _m['sam'].predict(**kw)
    return _to_size(np.asarray(m)[0] > 0.5, size)


def load_both():
    """파인튜닝 SAM + 원래 SAM을 둘 다 불러 둠"""
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    ft = load(NUKKI_WEIGHTS)
    _m['sam_ft'] = _m['sam'] if ft else None
    _m['sam_base'] = SAM2ImagePredictor.from_pretrained(SAM_MODEL, device=_m['device'])
    return ft


def use_sam(finetuned):
    _m['sam'] = _m['sam_ft'] if finetuned else _m['sam_base']
    _m['finetuned'] = finetuned
    _m['sam_img'] = None


HAS_FT = load_both()
print('device:', _m['device'], '| 파인튜닝 가중치:', HAS_FT)
if not HAS_FT:
    NUKKI_CONDS = [c for c in NUKKI_CONDS if c != 'dino_sam_ft']

## 1-2. RMBG-2.0 (비교용, 선택)
* 원래 계획의 분할 모델. 상차림 사진에서 테이블까지 잡는 문제로 DINO + SAM으로 바꿈 → 그 근거를 같은 사진으로 보여 주는 비교
* 라이선스: 비상업용(모델 페이지에서 확인). 서비스에는 쓰지 않고 **평가에서만** 씀

In [ ]:
rmbg = None
if USE_RMBG and 'rmbg' in NUKKI_CONDS:
    try:
        from google.colab import userdata
        from huggingface_hub import login
        from transformers import AutoModelForImageSegmentation
        from torchvision import transforms
        login(token=userdata.get('HF_TOKEN'))
        rmbg = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True).to(_m['device']).eval()
        rmbg_tf = transforms.Compose([transforms.Resize((1024, 1024)), transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    except Exception as e:
        print('RMBG를 건너뜀:', type(e).__name__, e)
if rmbg is None:
    NUKKI_CONDS = [c for c in NUKKI_CONDS if c != 'rmbg']


@torch.no_grad()
def rmbg_mask(img):
    x = rmbg_tf(img).unsqueeze(0).to(_m['device'], next(rmbg.parameters()).dtype)
    soft = rmbg(x)[-1].sigmoid()[0, 0].float().cpu().numpy()
    m = np.asarray(Image.fromarray((soft * 255).astype(np.uint8)).resize(img.size, Image.BILINEAR)) > 127
    return Image.fromarray((m * 255).astype(np.uint8))

print('A조건:', NUKKI_CONDS)

## 1-3. A조건 마스크 세트 만들기
* `eval/masks/{조건}/{사진}.png` (흰색 = 음식, 원본 크기). 이미 있으면 건너뜀
* 생성팀은 이 마스크를 받아 **같은 `EVAL_GEN`**으로 생성 → 분할만 바뀐 비교

In [ ]:
def nukki_mask(img, cond):
    if cond == 'rmbg':
        return rmbg_mask(img)
    use_sam(cond == 'dino_sam_ft')
    return segment(img)


times = []
for cond in NUKKI_CONDS:
    os.makedirs(f'{EVAL_DIR}/masks/{cond}', exist_ok=True)
    for i, n in enumerate(NAMES):
        out = f'{EVAL_DIR}/masks/{cond}/{n}.png'
        if os.path.exists(out):
            continue
        img = ImageOps.exif_transpose(Image.open(IMAGES[n])).convert('RGB')
        t0 = time.time()
        nukki_mask(img, cond).save(out)
        times.append({'cond': cond, 'image': n, 'sec': time.time() - t0})
    print(cond, '완료')
if times:
    pd.DataFrame(times).to_csv(f'{EVAL_DIR}/nukki_time.csv', index=False)
    print(pd.DataFrame(times).groupby('cond').sec.mean().round(2))

## 1-4. ① 음식 마스크 품질
* IoU, 경계 BF(허용 거리 = 대각선의 0.75%), MAE. 1번 노트북과 같은 계산

In [ ]:
def to_eval(mask_img, size):
    return np.asarray(mask_img.convert('L').resize(size, Image.BILINEAR), dtype=np.float32) / 255


def boundary_f1(p, g, tol):
    pb, gb = p ^ ndimage.binary_erosion(p), g ^ ndimage.binary_erosion(g)
    if not pb.any() and not gb.any():
        return 1.0
    if not pb.any() or not gb.any():
        return 0.0
    prec = (ndimage.distance_transform_edt(~gb)[pb] <= tol).mean()
    rec = (ndimage.distance_transform_edt(~pb)[gb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-8))


def mask_scores(pred_img, gt_img):
    k = EVAL['eval_side'] / max(gt_img.size)
    size = (round(gt_img.width * k), round(gt_img.height * k))
    g = to_eval(gt_img, size) > 0.5
    p = to_eval(pred_img, size)
    pb = p > 0.5
    union = (pb | g).sum()
    return {'IoU': float((pb & g).sum() / union) if union else 1.0,
            'BF': boundary_f1(pb, g, EVAL['bf_tol'] * np.hypot(*g.shape)),
            'MAE': float(np.abs(p - g).mean())}


rows = []
for n in NAMES:
    gt = Image.open(f"{GT_DIR}/{EVAL['split']}/{n}.png")
    for cond in NUKKI_CONDS:
        mp = f'{EVAL_DIR}/masks/{cond}/{n}.png'
        if os.path.exists(mp):
            rows.append({'image': n, 'category': CAT.get(n, '?'), 'cond': cond, **mask_scores(Image.open(mp), gt)})
mq = pd.DataFrame(rows)
mq.to_csv(f'{EVAL_DIR}/nukki_scores.csv', index=False)
print('=== ① 마스크 품질 (평균) ===')
print(mq.groupby('cond')[['IoU', 'BF', 'MAE']].mean().reindex(NUKKI_CONDS).round(4))
print('\n=== 분류별 IoU ===')
print(mq.pivot_table(index='category', columns='cond', values='IoU').reindex(columns=NUKKI_CONDS).round(3))

## 1-5. 샘플 그림 (보고서용 전후 비교)

In [ ]:
def tint(img_np, m, color=(255, 0, 0)):
    out = img_np.copy()
    out[m] = (out[m] * 0.5 + np.array(color) * 0.5).astype(np.uint8)
    return out


def plot_compare(names, save_name):
    cols = ['image', 'GT'] + NUKKI_CONDS
    fig, axes = plt.subplots(len(names), len(cols), figsize=(3.2 * len(cols), 3.4 * len(names)), squeeze=False)
    for r, n in enumerate(names):
        img = ImageOps.exif_transpose(Image.open(IMAGES[n])).convert('RGB')
        img.thumbnail((EVAL['eval_side'], EVAL['eval_side']))
        base = np.array(img)
        for c, col in enumerate(cols):
            ax = axes[r, c]
            if col == 'image':
                ax.imshow(base); ax.set_title(f'{n}\n[{CAT.get(n, "?")}]', fontsize=8)
            else:
                p = f"{GT_DIR}/{EVAL['split']}/{n}.png" if col == 'GT' else f'{EVAL_DIR}/masks/{col}/{n}.png'
                m = to_eval(Image.open(p), img.size) > 0.5
                ax.imshow(tint(base, m, (0, 200, 0) if col == 'GT' else (255, 0, 0)))
                s = mq[(mq.image == n) & (mq.cond == col)]
                ax.set_title(col if s.empty else f'{col}\nIoU {s.IoU.iloc[0]:.3f}', fontsize=8)
            ax.axis('off')
    plt.tight_layout(); plt.savefig(f'{EVAL_DIR}/{save_name}', dpi=120, bbox_inches='tight'); plt.show()


# 분류마다 대표 1~2장
pick = [n for c in ('white', 'multi', 'hand_tray', 'normal') for n in [x for x in NAMES if CAT.get(x) == c][:2]] or NAMES[:6]
plot_compare(pick, 'nukki_compare.png')

## 1-6. 다음 섹션에 넘기는 것
| 파일 | 내용 |
|---|---|
| `eval/masks/{rmbg, dino_sam, dino_sam_ft}/{사진}.png` | 조건별 음식 마스크 (흰색 = 음식, 원본 크기) |
| `eval/cases.csv` | 평가 케이스 (사진, 분류, 상충 종류, 요청 문장) |
| `eval/nukki_scores.csv` | ① 조건별·사진별 마스크 품질 |

# 2. VLM 평가 — VLM팀

* **B조건**: 마스크는 `eval/masks/dino_sam_ft` 고정, 프롬프트만 (a) `cases.csv`의 요청 그대로 / (b) VLM 해석 결과
* **③ 요청 반영** 측정 방법 정하기 (예: CLIP 이미지-문장 유사도, VLM 체크리스트 채점)

| 입력 | 출력 (저장 위치) |
|---|---|
| `eval/cases.csv`, 원본 사진 | `eval/prompts/{case}.json` (VLM이 만든 생성 프롬프트) |
| 생성 결과 `eval/gen/*/{case}.png` | `eval/request_scores.csv` (조건별 ③ 점수) |

In [ ]:
# VLM팀 작성

# 3. 배경 생성 평가 — 생성팀

* 모든 조건에서 `EVAL_GEN` 설정 고정
* **A조건**: `eval/masks/{조건}` 마스크 × 같은 프롬프트 → `eval/gen/A_{조건}/{case}.png`
* **B조건**: `dino_sam_ft` 마스크 × (요청 그대로 / VLM) → `eval/gen/B_raw/`, `eval/gen/B_vlm/`
* **C조건**: 전체 파이프라인, 원본 음식 재합성 켬/끔 → `eval/gen/C_full/`, `eval/gen/C_no_paste/`
* **② 픽셀 보존**: 음식 영역 안에서 원본과 같은 픽셀 비율 등. **정답 마스크(`dataset/gt/test`) 기준**으로 재면 어떤 분할을 썼든 같은 잣대로 비교 가능 → `eval/gen_scores.csv`

In [ ]:
# 생성팀 작성

# 4. 종합 — 다 같이

* **④ 시각적 품질**: 조건 이름을 가린 평가표를 만들어 2명 이상이 1~5점 → `eval/human_rating.csv`
* ①~④ 점수를 조건별 한 표로 모아서, **어느 단계가 결과를 개선했는지** 정리
* 보고서용: 같은 케이스의 A/B/C 결과를 나란히 놓은 그림

In [ ]:
# 종합 표, 그림